# T3P bandit — arm sweeps and runs

**Status: active.** T3P is the live selection mechanism; `dbs_t3p.py` is the module.

Split out of the old `dbs_rerun.ipynb`, whose first half was the abandoned TD3 re-run
(now `archive/td3_rerun.ipynb`). These cells were self-contained — they start their own
MATLAB backend and define their own config — so nothing was lost in the split.

What's here: full arm sweeps across the frequency and amplitude grid, T3P runs at several
severities, and comparisons between mild and severe brains.

**Watch the grid.** `dbs_t3p.CURR_VALUES` defaults to starting at 1000 µA, which skips the
300–550 µA region where the selector work found the useful settings. Several cells below
override it. Check which grid a result came from before quoting it, and check whether the
selected arm sits at a grid edge — this project has produced boundary artifacts repeatedly.

Run from the repo root so MATLAB writes `bgn_vars.mat` where the code expects it.

In [ ]:
import helpers, dbs_t3p as t3p

backend = t3p.MatlabBackend(code_dir=".", parent_dir=".")
cfg = t3p.T3PConfig(pd_value=1.0, tmax=1000)

rows = t3p.sweep_all_arms(backend, cfg, helpers, repeats=5)

In [ ]:
cfg = t3p.T3PConfig(pd_value=1.0, tmax=1000)
res, bandit = t3p.run_t3p(backend, cfg, helpers)

In [ ]:
results = []
for pd in [0.1, 0.4, 0.7, 1.0]:
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000, tag="v1")
    res, _ = t3p.run_t3p(backend, cfg, helpers)
    results.append(res)

t3p.compare_severities(results)

In [ ]:
import numpy as np
for pd in [0.0, 0.2, 0.5, 1.0]:
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    pbs = [helpers.calculate_pb_absolute(backend.simulate(130, 0, cfg)[0]) for _ in range(3)]
    print(f"pd={pd}: untreated pb = {np.mean(pbs):>8.0f} (sd {np.std(pbs):>6.0f})")

In [ ]:
import scipy.io
for pd in [0.0, 1.0]:
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    backend.eng.bgn_init(float(pd), float(cfg.tmax), 12345, nargout=0)
    d = scipy.io.loadmat("bgn_vars.mat")
    print(f"requested pd={pd} -> stored pd={float(d['pd'].flatten()[0])}")

In [ ]:
import numpy as np
for pd in [0.0, 1.0]:
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    pbs = [helpers.calculate_pb_absolute(backend.simulate(130, 0, cfg)[0]) for _ in range(10)]
    print(f"pd={pd}: {np.mean(pbs):>8.0f} +/- {np.std(pbs)/np.sqrt(10):>6.0f} (n=10)")

In [ ]:
import numpy as np
res = {}
for pd in [0.0, 1.0]:
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    a, b = [], []
    for _ in range(6):
        vgi, _ = backend.simulate(130, 0, cfg)
        a.append(helpers.calculate_pb_absolute(vgi))
        b.append(helpers.calculate_pb(vgi))
    res[pd] = (np.mean(a), np.std(a), np.mean(b), np.std(b))

for name, i in [("calculate_pb_absolute", 0), ("calculate_pb", 2)]:
    h, p = res[0.0][i], res[1.0][i]
    sh, sp = res[0.0][i+1], res[1.0][i+1]
    print(f"{name:<22} healthy {h:>9.0f} +/- {sh:>7.0f} | "
          f"PD {p:>9.0f} +/- {sp:>7.0f} | ratio {p/h:.2f}")

In [ ]:
from pathlib import Path
needed = ["th_minf","th_hinf","th_pinf","th_rinf","th_tauh","th_taur",
          "stn_minf","stn_ninf","stn_hinf","stn_ainf","stn_binf","stn_rinf",
          "stn_cinf","stn_taun","stn_tauh","stn_taur","stn_tauc",
          "gpe_minf","gpe_ninf","gpe_hinf","gpe_ainf","gpe_sinf","gpe_rinf",
          "gpe_taun","gpe_tauh","Hinf"]
have = {p.stem for p in Path("gating").glob("*.m")}
missing = [f for f in needed if f not in have]
print(f"{len(have)} .m files in gating/")
print("missing:", missing or "none")

In [ ]:
import numpy as np
import helpers
import dbs_t3p as t3p

print("t3p version:", t3p.__version__)
assert t3p.__version__ == "3.1.0", "old dbs_t3p.py — replace the file"

backend = t3p.MatlabBackend(code_dir=".", parent_dir=".")

In [ ]:
from pathlib import Path
root = Path.cwd()
backend.eng.addpath(str(root / "matlab"), nargout=0)
backend.eng.addpath(str(root / "gating"), nargout=0)
print(backend.eng.which("bgn_init"))
print(backend.eng.which("stn_ninf"))
cfg = t3p.T3PConfig(pd_value=1.0, tmax=1000)
rows = t3p.sweep_all_arms(backend, cfg, helpers, repeats=1)

In [ ]:
results = []
for pd in [0.1, 0.4, 0.7, 1.0]:
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000, tag="v2")
    res, _ = t3p.run_t3p(backend, cfg, helpers)
    results.append(res)

t3p.compare_severities(results)

In [ ]:
results = []
for pd in [0.1, 0.4, 0.7, 1.0]:
    for seed in [1, 2, 3]:
        cfg = t3p.T3PConfig(pd_value=pd, tmax=1000, seed=seed, tag=f"s{seed}")
        res, _ = t3p.run_t3p(backend, cfg, helpers, verbose=False)
        results.append(res)
        print(f"pd={pd} seed={seed}: {res['best_freq']}Hz/{res['best_curr']} Q={res['best_Q']:+.4f}")

In [ ]:
t3p.CURR_VALUES = [0, 100, 250, 500, 750, 1000, 2000, 3500, 5000]
t3p.ARMS = t3p.build_arms()
cfg = t3p.T3PConfig(pd_value=0.1, tmax=1000, n_arms=len(t3p.ARMS))
rows = t3p.sweep_all_arms(backend, cfg, helpers, repeats=3)

In [ ]:
import json
from pathlib import Path
for p in sorted(Path("runs_t3p").glob("*_s*.json")):
    r = json.loads(p.read_text())
    print(f"pd={r['config']['pd_value']} seed={r['config']['seed']}: "
          f"no-DBS {'KEPT' if 0 in r['active'] else 'PRUNED'}")

In [ ]:
cfg = t3p.T3PConfig(pd_value=1.0, tmax=1000, n_arms=len(t3p.ARMS))
rows_severe = t3p.sweep_all_arms(backend, cfg, helpers, repeats=3)

In [ ]:
import numpy as np
probe = [(180, 500), (180, 750), (180, 1000)]
for pd in (0.1, 1.0):
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    print(f"\npd={pd}")
    for f, a in probe:
        rs = [t3p.compute_reward(*backend.simulate(f, a, cfg), f, a, cfg, helpers)["reward"]
              for _ in range(10)]
        print(f"  {f}Hz/{a:<5} {np.mean(rs):+.4f} +/- {np.std(rs)/np.sqrt(10):.4f}")

In [ ]:
import numpy as np
for pd in (0.1, 1.0):
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    print(f"\npd={pd}")
    for a in (250, 300, 350, 400, 450, 500):
        rs = [t3p.compute_reward(*backend.simulate(180, a, cfg), 180, a, cfg, helpers)["reward"]
              for _ in range(10)]
        print(f"  180Hz/{a:<4} {np.mean(rs):+.4f} +/- {np.std(rs)/np.sqrt(10):.4f}")

In [ ]:
import numpy as np
curves = {}
for pd in (0.1, 0.4, 0.7, 1.0):
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    row = []
    for a in (250, 300, 350, 400, 450, 500):
        rs = [t3p.compute_reward(*backend.simulate(180, a, cfg), 180, a, cfg, helpers)["reward"]
              for _ in range(10)]
        row.append((a, float(np.mean(rs)), float(np.std(rs)/np.sqrt(10))))
    curves[pd] = row
    print(f"pd={pd}: " + "  ".join(f"{a}:{m:+.4f}" for a, m, _ in row))

def crossing(row, floor):
    for i in range(1, len(row)):
        if row[i-1][1] < floor <= row[i][1]:
            f = (floor - row[i-1][1]) / (row[i][1] - row[i-1][1])
            return row[i-1][0] + f * (row[i][0] - row[i-1][0])
    return None

print(f"\n{'floor':>8}" + "".join(f"{pd:>10}" for pd in curves))
for floor in (-0.05, -0.04, -0.03):
    line = f"{floor:>8.2f}"
    for pd in curves:
        c = crossing(curves[pd], floor)
        line += f"{c:>9.0f}u" if c else f"{'—':>10}"
    print(line)

In [ ]:
for pd in (0.1, 1.0):
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    rs0 = [t3p.compute_reward(*backend.simulate(130, 0, cfg), 130, 0, cfg, helpers)["reward"]
           for _ in range(5)]
    print(f"\npd={pd} | no DBS {np.mean(rs0):+.4f}")
    for a in (50, 100, 150, 200, 250):
        rs = [t3p.compute_reward(*backend.simulate(180, a, cfg), 180, a, cfg, helpers)["reward"]
              for _ in range(5)]
        flag = " worse than no DBS" if np.mean(rs) < np.mean(rs0) else ""
        print(f"  180Hz/{a:<4} {np.mean(rs):+.4f}{flag}")

In [ ]:
import numpy as np

AMPS = [250, 300, 350, 400, 450, 500]
PDS = [0.1, 0.4, 0.7, 1.0]
N = 20

curves = {}
for pd in PDS:
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    row = []
    for a in AMPS:
        rs = [t3p.compute_reward(*backend.simulate(180, a, cfg), 180, a, cfg, helpers)["reward"]
              for _ in range(N)]
        row.append((a, float(np.mean(rs)), float(np.std(rs, ddof=1) / np.sqrt(N))))
    curves[pd] = row
    print(f"pd={pd}: " + "  ".join(f"{a}:{m:+.4f}±{s:.4f}" for a, m, s in row))

print(f"\n{'amp':>5} " + "".join(f"{f'pd={p}':>18}" for p in PDS) + "   monotonic?")
for i, a in enumerate(AMPS):
    vals = [curves[p][i][1] for p in PDS]
    ses = [curves[p][i][2] for p in PDS]
    ok = all(vals[j] >= vals[j+1] for j in range(len(vals)-1))
    line = f"{a:>5} " + "".join(f"{v:>11.4f}±{s:.4f}" for v, s in zip(vals, ses))
    print(line + ("   yes" if ok else "   NO"))

print("\npd=0.7 vs pd=1.0 head to head:")
for i, a in enumerate(AMPS):
    m7, s7 = curves[0.7][i][1], curves[0.7][i][2]
    m10, s10 = curves[1.0][i][1], curves[1.0][i][2]
    d = m7 - m10
    print(f"  {a:>4}u  diff {d:+.4f}  ({abs(d)/np.hypot(s7, s10):.1f} sigma)"
          + ("  <- inverted" if d < 0 else ""))

In [ ]:
print(f"\n{'pd':>6} {'reward @180Hz/450':>22} {'pb':>12}")
for pd in [0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.9]:
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    recs = [t3p.compute_reward(*backend.simulate(180, 450, cfg), 180, 450, cfg, helpers)
            for _ in range(15)]
    r = [x["reward"] for x in recs]
    pb = [x["pb"] for x in recs]
    print(f"{pd:>6} {np.mean(r):>14.4f}±{np.std(r, ddof=1)/np.sqrt(15):.4f} "
          f"{np.mean(pb):>12.0f}")

In [ ]:
import numpy as np

AMPS = [250, 300, 350, 450]
PDS = [0.1, 1.0]
FREQS = [130, 155, 180]
N = 15

out = {}
for f in FREQS:
    print(f"\n=== {f} Hz ===")
    for pd in PDS:
        cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
        row = []
        for a in AMPS:
            rs = [t3p.compute_reward(*backend.simulate(f, a, cfg), f, a, cfg, helpers)["reward"]
                  for _ in range(N)]
            row.append((a, float(np.mean(rs)), float(np.std(rs, ddof=1)/np.sqrt(N))))
        out[(f, pd)] = row
        print(f"  pd={pd}: " + "  ".join(f"{a}:{m:+.4f}±{s:.4f}" for a, m, s in row))

print(f"\n{'freq':>6} {'amp':>6} {'mild':>10} {'severe':>10} {'gap':>9} {'sigma':>7}")
print("-" * 54)
for f in FREQS:
    for i, a in enumerate(AMPS):
        m, sm = out[(f, 0.1)][i][1], out[(f, 0.1)][i][2]
        s, ss = out[(f, 1.0)][i][1], out[(f, 1.0)][i][2]
        print(f"{f:>6} {a:>6} {m:>10.4f} {s:>10.4f} {m-s:>+9.4f} "
              f"{abs(m-s)/np.hypot(sm, ss):>7.1f}")

In [ ]:
import numpy as np

for pd in (0.1, 1.0):
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000)
    base = [t3p.compute_reward(*backend.simulate(130, 0, cfg), 130, 0, cfg, helpers)["pb"]
            for _ in range(15)]
    b, bse = np.mean(base), np.std(base, ddof=1)/np.sqrt(15)
    print(f"\npd={pd} | no DBS pb = {b:,.0f} +/- {bse:,.0f}")
    print(f"{'amp':>6} {'pb':>12} {'vs no-DBS':>12} {'sigma':>7}")
    for a in (150, 200, 250, 275, 300, 325, 350):
        v = [t3p.compute_reward(*backend.simulate(180, a, cfg), 180, a, cfg, helpers)["pb"]
             for _ in range(15)]
        m, se = np.mean(v), np.std(v, ddof=1)/np.sqrt(15)
        sig = (m - b) / np.hypot(se, bse)
        tag = "  HARMFUL" if sig > 2 else ("  helps" if sig < -2 else "  same")
        print(f"{a:>6} {m:>12,.0f} {m-b:>+12,.0f} {sig:>7.1f}{tag}")

In [ ]:
print(f"\n{'seed':>6} {'mild@450':>12} {'severe@450':>12} {'gap':>9}")
for seed in (11, 22, 33):
    out = {}
    for pd in (0.1, 1.0):
        cfg = t3p.T3PConfig(pd_value=pd, tmax=1000, seed=seed)
        v = [t3p.compute_reward(*backend.simulate(155, 450, cfg), 155, 450, cfg, helpers)["reward"]
             for _ in range(12)]
        out[pd] = np.mean(v)
    print(f"{seed:>6} {out[0.1]:>12.4f} {out[1.0]:>12.4f} {out[0.1]-out[1.0]:>+9.4f}")